In [1]:
import json
import os

from pathlib import Path
from google import genai

In [2]:
GEMINI_API_KEY = "AQ.Ab8RN6J-CkEiqCak93fgtFqs_X2XoJ58G0b5Jx0JL2_s5Fx5IQ"

client = genai.Client(api_key=GEMINI_API_KEY)

In [4]:
#Load navigation index
import json

with open(
    "LuatLaoDong2019_tree.json",
    "r",
    encoding="utf-8"
) as f:

    document = json.load(f)

nodes = document["nodes"]

node_index = {
    node["node_id"]: node
    for node in nodes
}

with open(
    "LuatLaoDong2019_navigation.txt",
    "r",
    encoding="utf-8"
) as f:

    tree_outline = f.read()

print(f"Loaded {len(nodes)} nodes.")
print(f"Indexed {len(node_index)} nodes.")
print(f"Navigation: {len(tree_outline):,} characters")
print(f"Navigation: {len(tree_outline.split()):,} words")

Loaded 1169 nodes.
Indexed 1169 nodes.
Navigation: 141,336 characters
Navigation: 26,047 words


In [18]:
query = "Nam giới được nghỉ hưu mấy tuổi?"

In [7]:
query = "Lái xe đội nón phạt bao nhiêu?"

In [19]:
#Prompt for tree search
def create_node_selector_prompt(query, tree_outline):
    prompt = f"""
Bạn là hệ thống tìm kiếm văn bản pháp luật Việt Nam.

Nhiệm vụ:
Dựa trên cây cấu trúc của văn bản, chọn các node liên quan nhất với câu hỏi.

KHÔNG trả lời câu hỏi.

Quy tắc:

- Chọn từ 0 đến 5 node.
- Ưu tiên node cụ thể nhất.
- Không chọn node cha nếu node con đã chứa đầy đủ thông tin.
- Chỉ chọn node cha khi cần thêm ngữ cảnh.
- Chỉ trả về JSON hợp lệ.
- Không giải thích ngoài JSON.

Output JSON duy nhất:
[
  {{
    "node_id": 123,
    "thinking": "giải thích lý do chọn node"
  }}
]

CÂY VĂN BẢN:
{tree_outline}

CÂU HỎI:
{query}
"""

    return prompt

In [9]:
from google.genai import types

def select_nodes(query, tree_outline):
    prompt = create_node_selector_prompt(
        query=query,
        tree_outline=tree_outline
    )

    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json",
        )
    )

    return response.text

In [20]:
#Test tree search
result = select_nodes(
    query=query,
    tree_outline=tree_outline
)

print(result)

[
  {
    "node_id": 855,
    "thinking": "Khoản 2 Điều 169 quy định về tuổi nghỉ hưu của người lao động trong điều kiện lao động bình thường, bao gồm lộ trình điều chỉnh tuổi nghỉ hưu đối với lao động nam và lao động nữ."
  }
]


In [21]:
selected = json.loads(result)

print(selected)

[{'node_id': 855, 'thinking': 'Khoản 2 Điều 169 quy định về tuổi nghỉ hưu của người lao động trong điều kiện lao động bình thường, bao gồm lộ trình điều chỉnh tuổi nghỉ hưu đối với lao động nam và lao động nữ.'}]


In [22]:
#Fetch selected note full data
def fetch_selected_nodes(selected):
    result = []
    for item in selected:
        node = node_index.get(item["node_id"])

        if node is None:
            continue

        result.append({
            "reason": item.get("reason", ""),
            "node": node
        })

    return result

In [23]:
selected_nodes = fetch_selected_nodes(selected)
for item in selected_nodes:
    node = item["node"]

    print("=" * 60)
    print(f"Node ID : {node['node_id']}")
    print(f"Type    : {node['type']}")
    print(f"Number  : {node['number']}")
    print(f"Title   : {node['title']}")
    print()
    print(node["content"][:300])

Node ID : 855
Type    : KHOAN
Number  : 2
Title   : Khoản 2

Tuổi nghỉ hưu của người lao động trong điều kiện lao động bình thường được điều chỉnh theo lộ trình cho đến khi đủ 62 tuổi đối với lao động nam vào năm 2028 và đủ 60 tuổi đối với lao động nữ vào năm 2035.
Kể từ năm 2021, tuổi nghỉ hưu của người lao động trong điều kiện lao động bình thường là đủ 60 


In [24]:
#Build Context for answer LLM
def build_context(selected_nodes):

    if not selected_nodes:
        return ""

    sections = []

    for item in selected_nodes:

        node = item["node"]
        lines = []

        #Hierarchy
        for p in node["path"]:

            if p["type"] == "CHUONG":
                lines.append(
                    f"Chương {p['number']} - {p['title']}"
                )

            elif p["type"] == "DIEU":
                lines.append(
                    f"Điều {p['number']} - {p['title']}"
                )

            elif p["type"] == "KHOAN":
                lines.append(
                    f"Khoản {p['number']}"
                )

            elif p["type"] == "DIEM":
                lines.append(
                    f"Điểm {p['number']}"
                )

        lines.append("")
        lines.append(node["content"])

        sections.append(
            "\n".join(lines)
        )

    separator = (
        "\n"
        + "=" * 80
        + "\n\n"
    )

    return separator.join(sections)

In [25]:
context = build_context(selected_nodes)

print(context)

Chương XII - BẢO HIỂM XÃ HỘI, BẢO HIỂM Y TẾ, BẢO HIỂM THẤT NGHIỆP
Điều 169 - Tuổi nghỉ hưu
Khoản 2

Tuổi nghỉ hưu của người lao động trong điều kiện lao động bình thường được điều chỉnh theo lộ trình cho đến khi đủ 62 tuổi đối với lao động nam vào năm 2028 và đủ 60 tuổi đối với lao động nữ vào năm 2035.
Kể từ năm 2021, tuổi nghỉ hưu của người lao động trong điều kiện lao động bình thường là đủ 60 tuổi 03 tháng đối với lao động nam và đủ 55 tuổi 04 tháng đối với lao động nữ; sau đó, cứ mỗi năm tăng thêm 03 tháng đối với lao động nam và 04 tháng đối với lao động nữ.


In [26]:
#Answer for query LLM
from google.genai import types

def answer_question(query, context):
    if not context.strip():
        return "Không biết, hoặc thông tin không có trong dữ liệu."

    prompt = f"""
Bạn là trợ lý hỏi đáp về pháp luật Việt Nam.

Chỉ được sử dụng thông tin trong CONTEXT để trả lời.

Quy tắc:

- Không tự suy diễn.
- Không bổ sung kiến thức bên ngoài.
- Nếu Context chứa nhiều thông tin hơn câu hỏi cần biết thì chỉ trả lời câu hỏi mà context có chứa
- Trả lời ngắn gọn, rõ ràng bằng tiếng Việt.
- Đầu câu trả lời hãy thêm "Theo bộ luật lao động Việt Nam 2019" Chương nào - Điều nào - Khoản nào (nếu có, không có không cần khai báo) - Điểm nào (nếu có, không có không cần khai báo).

CONTEXT:
{context}

QUESTION:
{query}
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.2
        )
    )

    return response.text.strip()

In [27]:
answer = answer_question(
    query=query,
    context=context
)

print(query)
print()
print(answer)

Nam giới được nghỉ hưu mấy tuổi?

Theo bộ luật lao động Việt Nam 2019 Chương XII - Điều 169 - Khoản 2: 

Trong điều kiện lao động bình thường, tuổi nghỉ hưu của lao động nam được điều chỉnh theo lộ trình như sau:
- Kể từ năm 2021 là đủ 60 tuổi 03 tháng, sau đó cứ mỗi năm tăng thêm 03 tháng.
- Cho đến khi đủ 62 tuổi vào năm 2028.
